In [ ]:
import pandas as pd
import re
import time
import musicbrainzngs
import requests

In [ ]:
df = pd.read_csv('mirdei_lyrics.csv')

def clean_lyrics(text):
    if pd.isna(text):
        return text
    
    # Remove annotations in square brackets
    text = re.sub(r'\[.*?\]', '', text)
    
    # Remove blank lines and extra newlines
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    text = ' '.join(lines)
    
    return text

df['Lyrics'] = df['Lyrics'].apply(clean_lyrics)
df.to_csv('mirdei_lyrics_clean.csv', index=False)

Cleaned 543 songs


In [ ]:
num_artists = df['Artist'].nunique()
print(f"Number of different artists: {num_artists}")
print(f"\nTop 10 artists by song count:")
print(df['Artist'].value_counts().head(10))

Number of different artists: 423

Top 10 artists by song count:
Artist
Billie Holiday            9
Gipsy Kings               6
Ella Fitzgerald           5
Dead Can Dance            5
Willie Nelson             4
Discharge                 4
Einstürzende Neubauten    4
Gloria Estefan            4
Onyx                      4
Jimmy Reed                3
Name: count, dtype: int64


In [ ]:
# musicBrainz useragent (required by their API)
musicbrainzngs.set_useragent("MyMusicApp", "1.0", "contact@example.com")

def search_musicbrainz_id(artist, title):
    try:
        result = musicbrainzngs.search_recordings(artist=artist, recording=title, limit=1)
        
        if result['recording-count'] > 0:
            recording_id = result['recording-list'][0]['id']
            return recording_id
        else:
            return None
    
    except Exception as e:
        print(f"Error searching MusicBrainz: {e}")
        return None

In [ ]:
# Add column for recording IDs
df['MusicBrainz_ID'] = None

print("Extracting MusicBrainz recording IDs for all songs...")
print(f"Total songs to process: {len(df)}\n")

successful = 0
for idx, row in df.iterrows():
    if idx % 100 == 0:
        print(f"[{idx}/{len(df)}] Processing... ({(idx/len(df)*100):.1f}%)")
    
    artist = row['Artist']
    title = row['Title']
    
    recording_id = search_musicbrainz_id(artist, title)
    df.at[idx, 'MusicBrainz_ID'] = recording_id
    
    if recording_id is not None:
        successful += 1
    
    # rate limiting
    time.sleep(0.2)

print(f"\n{'='*50}")
print(f"Completed extracting MusicBrainz IDs")
print(f"{'='*50}")
print(f"Total songs processed: {len(df)}")
print(f"Songs with MusicBrainz ID: {df['MusicBrainz_ID'].notna().sum()}")

Extracting MusicBrainz recording IDs for all songs...
Total songs to process: 543

[0/543] Processing... (0.0%)
[100/543] Processing... (18.4%)
[100/543] Processing... (18.4%)
[200/543] Processing... (36.8%)
[200/543] Processing... (36.8%)
[300/543] Processing... (55.2%)
[300/543] Processing... (55.2%)
[400/543] Processing... (73.7%)
[400/543] Processing... (73.7%)
[500/543] Processing... (92.1%)
[500/543] Processing... (92.1%)

Completed extracting MusicBrainz IDs
Total songs processed: 543
Songs with MusicBrainz ID: 543
Success rate: 100.0%

Saved to 'mirdei_lyrics_with_ids.csv'

Completed extracting MusicBrainz IDs
Total songs processed: 543
Songs with MusicBrainz ID: 543
Success rate: 100.0%

Saved to 'mirdei_lyrics_with_ids.csv'


In [ ]:
def get_acousticbrainz_features(recording_id):
    try:
        ab_url = f"https://acousticbrainz.org/api/v1/{recording_id}/low-level"
        response = requests.get(ab_url, timeout=5)
        response.raise_for_status()
        
        data = response.json()
        
        key = data.get('tonal', {}).get('key_key', None)
        mode = data.get('tonal', {}).get('key_scale', None)
        
        return key, mode
    
    except Exception as e:
        return None, None

Testing AcousticBrainz with:
Artist: Dismember
Title: Reborn in Blasphemy
Recording ID: 11680f99-e105-46c1-a99a-800dd8cc685b

Fetching features from AcousticBrainz...
Found features!
Key: C
Mode: minor
Found features!
Key: C
Mode: minor


In [ ]:
# Add columns for key and mode
df['Key'] = None
df['Mode'] = None

print("Extracting key and mode from AcousticBrainz for all songs...")
print(f"Total songs to process: {len(df)}\n")

successful = 0
for idx, row in df.iterrows():
    if idx % 100 == 0:
        print(f"[{idx}/{len(df)}] Processing... ({(idx/len(df)*100):.1f}%)")
    
    recording_id = row['MusicBrainz_ID']
    
    if pd.notna(recording_id):
        key, mode = get_acousticbrainz_features(recording_id)
        df.at[idx, 'Key'] = key
        df.at[idx, 'Mode'] = mode
        
        if key is not None or mode is not None:
            successful += 1
    
    # rate limiting
    time.sleep(0.1)

print(f"\n{'='*50}")
print(f"Completed extracting features from AcousticBrainz")
print(f"{'='*50}")
print(f"Total songs processed: {len(df)}")
print(f"Songs with Key: {df['Key'].notna().sum()}")
print(f"Songs with Mode: {df['Mode'].notna().sum()}")
print(f"Songs with either Key or Mode: {successful}")
print(f"Success rate: {(successful/len(df)*100):.1f}%")

# moving lyrics column to the end
lyrics_col = df.pop('Lyrics')
df['Lyrics'] = lyrics_col

df.to_csv('mirdei_lyrics_acousticbrainz.csv', index=False)

Extracting key and mode from AcousticBrainz for all songs...
Total songs to process: 543

[0/543] Processing... (0.0%)
[100/543] Processing... (18.4%)
[100/543] Processing... (18.4%)
[200/543] Processing... (36.8%)
[200/543] Processing... (36.8%)
[300/543] Processing... (55.2%)
[300/543] Processing... (55.2%)
[400/543] Processing... (73.7%)
[400/543] Processing... (73.7%)
[500/543] Processing... (92.1%)
[500/543] Processing... (92.1%)

Completed extracting features from AcousticBrainz
Total songs processed: 543
Songs with Key: 287
Songs with Mode: 287
Songs with either Key or Mode: 287
Success rate: 52.9%

Saved to 'mirdei_lyrics_with_features.csv'

Completed extracting features from AcousticBrainz
Total songs processed: 543
Songs with Key: 287
Songs with Mode: 287
Songs with either Key or Mode: 287
Success rate: 52.9%

Saved to 'mirdei_lyrics_with_features.csv'


In [ ]:
missing_ids = df['MusicBrainz_ID'].isna().sum()
missing_features = df[['Key', 'Mode']].isna().all(axis=1).sum()
missing_with_id = df[df['MusicBrainz_ID'].notna() & df[['Key', 'Mode']].isna().all(axis=1)]

print("Diagnostics after AcousticBrainz fetch:")
print(f"Total songs: {len(df)}")
print(f"Missing MusicBrainz IDs: {missing_ids}")
print(f"Rows with no key and mode: {missing_features}")
print(f"Rows with an ID but no key/mode: {len(missing_with_id)}")

if not missing_with_id.empty:
    display_cols = ['Artist', 'Title', 'MusicBrainz_ID', 'Key', 'Mode']
    print("\nSample rows with ID but missing key/mode:")
    print(missing_with_id[display_cols].head(10))
else:
    print("All rows with IDs have key or mode.")

Diagnostics after AcousticBrainz fetch:
Total songs: 543
Missing MusicBrainz IDs: 0
Rows with no key and mode: 256
Rows with an ID but no key/mode: 256

Sample rows with ID but missing key/mode:
                    Artist                                              Title  \
1            Little Walter                                         Last Night   
2           The Beach Boys                                  Little Saint Nick   
4               De La Soul                                           All Good   
5                 Deftones                                    Rats!Rats!Rats!   
10         The "5" Royales                              Let Me Come Back Home   
14             Dean Martin                   Volare (Nel Blu di Pinto di Blu)   
16              Eyehategod                            Take as Needed for Pain   
19  Gatsbys American Dream                               Station 5: The Pearl   
23       The Soul Stirrers  My Loved Ones Are Waiting for Me (Waiting and ..